In [7]:
from astropy.io import fits

data = fits.getdata("/home/bernd/broadband_397866.fits", 0)

In [8]:
hdul = fits.open("/home/bernd/broadband_397866.fits")
for i, hdu in enumerate(hdul):
    print(f"--- HDU {i}: {hdu.name} ---")
    print(repr(hdu.header))
    print()


--- HDU 0: PRIMARY ---
SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -32 / array data type                                
NAXIS   =                    3 / number of array dimensions                     
NAXIS1  =                   93                                                  
NAXIS2  =                   93                                                  
NAXIS3  =                    4                                                  
BUNIT   = 'counts/s/pixel'     / Unit of the array values                       
CDELT1  =    388.7452183439798 / Coordinate increment along X-axis              
CTYPE1  = 'pc      '           / Physical units of the X-axis increment         
CDELT2  =    388.7452183439798 / Coordinate increment along Y-axis              
CTYPE2  = 'pc      '           / Physical units of the Y-axis increment         
PIXSCALE=                0.396 / Pixel size in arcsec                           
USE_Z

In [9]:
import numpy as np
from astropy.modeling import models, fitting

# Use 2D image: sum over bands if 3D
img = data.sum(axis=0) if data.ndim == 3 else data

ny, nx = img.shape
y, x = np.mgrid[0:ny, 0:nx]

# Initial guesses
x0, y0 = nx / 2, ny / 2
amplitude = float(img.max())
r_eff = nx / 8

sersic_init = models.Sersic2D(
    amplitude=amplitude,
    r_eff=r_eff,
    n=2,
    x_0=x0,
    y_0=y0,
    ellip=0.0,
    theta=0.0,
)

fitter = fitting.LevMarLSQFitter()
fitted = fitter(sersic_init, x, y, img, maxiter=500)

print(f"Sersic index n = {fitted.n.value:.4f}")
fitted


Sersic index n = 1.2179


<Sersic2D(amplitude=70.08584691, r_eff=6.42702326, n=1.21785814, x_0=46.11635025, y_0=46.15567813, ellip=1.31096902, theta=8.58044065e+08)>